# Example: Multi-Backend Performance Comparison

Compare performance of Pandas, Polars, and DuckDB backends for factor calculation on the same dataset.

## Objectives
- Benchmark calculation speed across backends
- Verify numerical consistency
- Understand when to use each backend
- Memory usage comparison

In [ ]:
import sys
import numpy as np
import pandas as pd
import time
from typing import Dict

sys.path.insert(0, '/home/shw/quant_projects/factor_engine')
sys.path.insert(0, '/home/shw/quant_projects/notebooks')

from utils import display_success, display_warning, display_metrics, ProgressBar

print("Multi-Backend Performance Comparison")

## Step 1: Generate Test Dataset

Create a large dataset to stress-test backends.

In [ ]:
def generate_large_dataset(n_stocks=500, n_days=1000):
    """Generate large market dataset."""
    np.random.seed(42)
    
    print(f"Generating {n_stocks} stocks × {n_days} days = {n_stocks * n_days:,} observations")
    
    dates = pd.date_range(end='2024-01-01', periods=n_days, freq='B')
    tickers = [f'STOCK_{i:04d}' for i in range(n_stocks)]
    
    data = []
    for ticker in tickers:
        base_price = 50 + np.random.randn() * 20
        prices = base_price * np.exp(np.cumsum(np.random.randn(n_days) * 0.02))
        volume = np.abs(np.random.randn(n_days) * 1e6 + 5e6)
        
        for i, date in enumerate(dates):
            data.append({
                'date': date,
                'ticker': ticker,
                'close': prices[i],
                'open': prices[i] * (1 + np.random.randn() * 0.005),
                'high': prices[i] * (1 + abs(np.random.randn() * 0.01)),
                'low': prices[i] * (1 - abs(np.random.randn() * 0.01)),
                'volume': volume[i],
            })
    
    df = pd.DataFrame(data)
    memory_mb = df.memory_usage(deep=True).sum() / 1024**2
    
    print(f"Dataset memory: {memory_mb:.1f} MB")
    
    return df.set_index(['date', 'ticker']).sort_index()

test_data = generate_large_dataset(n_stocks=500, n_days=1000)

print(f"\nShape: {test_data.shape}")
print(f"Columns: {list(test_data.columns)}")

display_success("Test dataset ready")

## Step 2: Define Test Factors

Create a set of factors with varying complexity.

In [ ]:
from api import col, rank, ts_mean, ts_std, delay, zscore, Factor

# Define factors with different computational complexity
test_factors = [
    ('simple_rank', rank(col('close')), 'Simple cross-sectional rank'),
    ('momentum_20', rank(col('close') / delay(col('close'), 20) - 1), 'Short momentum'),
    ('ma_diff', rank(ts_mean(col('close'), 20) - ts_mean(col('close'), 5)), 'Moving average diff'),
    ('volatility', rank(ts_std(col('close'), 20)), 'Rolling volatility'),
    ('zscore_momentum', zscore(col('close') / delay(col('close'), 60) - 1), 'Z-scored momentum'),
]

print("Test Factors:")
for name, expr, desc in test_factors:
    print(f"  {name}: {desc}")

## Step 3: Benchmark Pandas Backend

Test all factors with the Pandas backend.

In [ ]:
from backend.pandas_backend import PandasBackend
from storage.datasource import DataSource
from runtime.engine import FactorEngine

class PandasDataSource(DataSource):
    def __init__(self, df):
        self.df = df
    def load_column(self, name: str):
        return self.df[name]

def benchmark_backend(backend_name: str, backend, data_source, factors):
    """Benchmark a backend with multiple factors."""
    engine = FactorEngine(backend=backend, data_source=data_source)
    results = {}
    
    print(f"\nBenchmarking {backend_name}...")
    progress = ProgressBar(len(factors), backend_name)
    
    for factor_name, factor_expr, desc in factors:
        factor = Factor(factor_name, factor_expr, '1d', 'equities')
        
        start = time.time()
        try:
            result = engine.run(factor)
            elapsed = time.time() - start
            
            factor_values = result['result']
            results[factor_name] = {
                'success': True,
                'time': elapsed,
                'coverage': factor_values.notna().sum() / len(factor_values),
                'values': factor_values,
            }
        except Exception as e:
            elapsed = time.time() - start
            results[factor_name] = {
                'success': False,
                'time': elapsed,
                'error': str(e),
            }
        
        progress.update(1)
    
    progress.close()
    return results

# Benchmark Pandas
pandas_source = PandasDataSource(test_data)
pandas_results = benchmark_backend('Pandas', PandasBackend(), pandas_source, test_factors)

print("\nPandas Results:")
for name, res in pandas_results.items():
    if res['success']:
        print(f"  {name}: {res['time']:.3f}s (coverage: {res['coverage']:.1%})")
    else:
        print(f"  {name}: FAILED - {res['error']}")

display_success("Pandas benchmark complete")

## Step 4: Benchmark Alternative Backends

Test Polars and DuckDB if available.

In [ ]:
# Try Polars backend
try:
    from backend.polars_backend import PolarsBackend
    polars_results = benchmark_backend('Polars', PolarsBackend(), pandas_source, test_factors)
    
    print("\nPolars Results:")
    for name, res in polars_results.items():
        if res['success']:
            print(f"  {name}: {res['time']:.3f}s (coverage: {res['coverage']:.1%})")
        else:
            print(f"  {name}: FAILED - {res['error']}")
    
    display_success("Polars benchmark complete")
except ImportError:
    print("Polars backend not available")
    polars_results = None

# Try DuckDB backend
try:
    from backend.duckdb_backend import DuckDBBackend
    duckdb_results = benchmark_backend('DuckDB', DuckDBBackend(), pandas_source, test_factors)
    
    print("\nDuckDB Results:")
    for name, res in duckdb_results.items():
        if res['success']:
            print(f"  {name}: {res['time']:.3f}s (coverage: {res['coverage']:.1%})")
        else:
            print(f"  {name}: FAILED - {res['error']}")
    
    display_success("DuckDB benchmark complete")
except ImportError:
    print("DuckDB backend not available")
    duckdb_results = None

## Step 5: Compare Performance

Aggregate and compare timing results.

In [ ]:
# Collect all results
all_results = {'Pandas': pandas_results}
if polars_results:
    all_results['Polars'] = polars_results
if duckdb_results:
    all_results['DuckDB'] = duckdb_results

# Build comparison table
comparison_data = []
for factor_name, _, _ in test_factors:
    row = {'factor': factor_name}
    for backend_name, results in all_results.items():
        if factor_name in results and results[factor_name]['success']:
            row[f'{backend_name}_time'] = results[factor_name]['time']
        else:
            row[f'{backend_name}_time'] = np.nan
    comparison_data.append(row)

comparison_df = pd.DataFrame(comparison_data)

print("\n=== PERFORMANCE COMPARISON ===")
print(comparison_df.to_string(index=False))

# Calculate speedups
if len(all_results) > 1:
    print("\n=== SPEEDUP vs PANDAS ===")
    for backend_name in all_results.keys():
        if backend_name != 'Pandas':
            speedups = []
            for _, row in comparison_df.iterrows():
                pandas_time = row['Pandas_time']
                backend_time = row.get(f'{backend_name}_time', np.nan)
                if not np.isnan(pandas_time) and not np.isnan(backend_time) and backend_time > 0:
                    speedup = pandas_time / backend_time
                    speedups.append(speedup)
            
            if speedups:
                avg_speedup = np.mean(speedups)
                print(f"  {backend_name}: {avg_speedup:.2f}x")
                if avg_speedup > 1.2:
                    display_success(f"{backend_name} is {avg_speedup:.2f}x faster")
                elif avg_speedup < 0.8:
                    display_warning(f"{backend_name} is slower: {1/avg_speedup:.2f}x")

## Step 6: Verify Numerical Consistency

Ensure backends produce the same results.

In [ ]:
def compare_factor_values(results_dict: Dict, factor_name: str, tolerance: float = 1e-6):
    """Compare factor values across backends."""
    backends = list(results_dict.keys())
    values = {}
    
    for backend in backends:
        if factor_name in results_dict[backend] and results_dict[backend][factor_name]['success']:
            values[backend] = results_dict[backend][factor_name]['values']
    
    if len(values) < 2:
        return None
    
    # Compare first backend to others
    base_backend = backends[0]
    base_values = values[base_backend]
    
    comparisons = {}
    for backend in backends[1:]:
        if backend in values:
            other_values = values[backend]
            
            # Align on common index
            common_idx = base_values.index.intersection(other_values.index)
            base_aligned = base_values.loc[common_idx].dropna()
            other_aligned = other_values.loc[common_idx].dropna()
            
            common_idx2 = base_aligned.index.intersection(other_aligned.index)
            
            if len(common_idx2) > 0:
                diff = (base_aligned.loc[common_idx2] - other_aligned.loc[common_idx2]).abs()
                max_diff = diff.max()
                mean_diff = diff.mean()
                
                comparisons[backend] = {
                    'max_diff': max_diff,
                    'mean_diff': mean_diff,
                    'match': max_diff < tolerance,
                    'n_compared': len(common_idx2),
                }
    
    return comparisons

print("\n=== NUMERICAL CONSISTENCY CHECK ===")
for factor_name, _, _ in test_factors:
    print(f"\n{factor_name}:")
    comparison = compare_factor_values(all_results, factor_name)
    
    if comparison:
        for backend, result in comparison.items():
            status = "✓ MATCH" if result['match'] else "✗ MISMATCH"
            print(f"  Pandas vs {backend}: {status}")
            print(f"    Max diff: {result['max_diff']:.2e}")
            print(f"    Mean diff: {result['mean_diff']:.2e}")
            print(f"    Samples: {result['n_compared']}")
            
            if not result['match']:
                display_warning(f"Numerical mismatch for {factor_name} on {backend}")
    else:
        print("  Insufficient data for comparison")

display_success("Consistency check complete")

## Step 7: Backend Recommendations

Summarize when to use each backend.

In [ ]:
print("\n=== BACKEND RECOMMENDATIONS ===")

print("\n**Pandas Backend:**")
print("  Use when:")
print("    - Dataset < 100k rows")
print("    - Complex custom operations")
print("    - Interactive exploration")
print("    - Maximum compatibility")
print("  Pros: Most mature, full feature support")
print("  Cons: Slower on large datasets")

if polars_results:
    print("\n**Polars Backend:**")
    print("  Use when:")
    print("    - Dataset 100k - 10M rows")
    print("    - Standard operations (rank, mean, std)")
    print("    - Memory efficiency matters")
    print("  Pros: Fast, memory efficient")
    print("  Cons: Some operators may not be implemented")

if duckdb_results:
    print("\n**DuckDB Backend:**")
    print("  Use when:")
    print("    - Dataset > 10M rows")
    print("    - Data larger than memory")
    print("    - SQL-friendly operations")
    print("  Pros: Handles huge datasets, out-of-core processing")
    print("  Cons: Some complex operators may not translate to SQL")

print("\n**General Rule:**")
print("  Start with Pandas for development, switch to Polars/DuckDB for production scale.")

display_success("Backend comparison complete")

## Summary

This example demonstrates:

1. **Benchmarking**: How to measure factor calculation time across backends
2. **Consistency**: Verifying numerical results match across implementations
3. **Scalability**: Understanding performance characteristics at scale
4. **Selection**: Choosing the right backend for your use case

## Key Findings

- **Pandas**: Best for development and datasets < 100k rows
- **Polars**: 2-5x faster for medium datasets, good memory efficiency
- **DuckDB**: Handles arbitrarily large data with out-of-core processing
- **Consistency**: All backends should produce numerically identical results

## Production Tips

1. Develop and test factors with Pandas on small samples
2. Switch to Polars for faster iteration on full datasets
3. Use DuckDB for historical backtests on years of data
4. Always verify numerical consistency when switching backends
5. Profile your specific workload - results vary by operation type